# Table 2, $T=6$, unrestricted Gaussian VI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tlamadon/ivi-replicate/blob/main/notebooks/table2_vi_gaussian.ipynb)

This notebook reproduces one line of **Table 2: Parameter Estimates in the
Nonlinear Model** of *Indirect Variational Inference: Applications to Earnings
Dynamics* (Balke, Bonhomme and Lamadon,
[arXiv:2607.15168](https://arxiv.org/abs/2607.15168)): the $T=6$ row
**(1) unrestr. Gaussian**, a variational fit with an unrestricted Gaussian
(joint-normal) posterior on a simulated panel of $N=30{,}000$ individuals.

It clones the [replication package](https://github.com/tlamadon/ivi-replicate),
runs the same script with the same settings and seeds as `replicate.py`
(`scripts/_simulation_hockeystick_ivi_sweep.py`, 20,000 epochs), and compares
the estimates with the published ones.

**Use a GPU runtime** (*Runtime → Change runtime type → GPU*). The fit takes
roughly 20–40 minutes on Colab's free T4 (under 5 minutes on a recent desktop GPU). GPU arithmetic is not bit-identical across devices, so the last
digit can differ from the published table.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

`PINNED = True` installs the exact package versions of the replication
environment (`requirements.lock`, including torch 2.13 / CUDA 13; a few GB to
download). If that fails on your runtime, set `PINNED = False` to keep Colab's
own torch and install only what is missing.

In [ ]:
PINNED = True

!git clone --depth 1 https://github.com/tlamadon/ivi-replicate.git
%cd ivi-replicate
if PINNED:
    !pip install -q -r requirements.lock
else:
    !pip install -q pydantic structlog nvidia-ml-py
!pip install -q --no-deps -e .

## 2. Run the estimation

Settings copied from `block_hockey` in `replicate.py` (the VI cells of the
hockey-stick simulation, unrestricted Gaussian encoder `ENCODER=jn`). In the
published run this fit is Phase 1 of the IVI cell, which the table also reports
as row (1). After the fit, the script computes ELBO/IWAE and SMC diagnostics
before writing its JSON.

In [ ]:
import os, subprocess, sys

env = dict(os.environ,
           SIGMA0="-1.8", SIGMA2="0.35", LR="0.01", LR_TAG="1e-2", LOG_EVERY="200",
           NOISE_SEED="12345", OBS_SEED="11",
           METHOD="vi_only", ENCODER="jn", FIX_NOISE="1", N_EPOCHS_VI_OBS="20000",
           OUT_DIR="colab_run")
subprocess.run([sys.executable, "scripts/_simulation_hockeystick_ivi_sweep.py"], env=env, check=True)

## 3. Compare with the published table

Parameters are mapped to the table's columns with the same function the paper's
table generator uses (`scripts/fig_hockeystick_parameter_table.py`); the
published values come from `output/bundles/hockeystick_parameter_summary.json`.

In [ ]:
import glob, json
from IPython.display import Markdown, display

sys.path.insert(0, "scripts")
import fig_hockeystick_parameter_table as tab

(cell_path,) = glob.glob("colab_run/vi_only_jn_*.json")
mine = json.load(open(cell_path))["theta_VI_obs"]

summary = tab.load_summary("output/bundles/hockeystick_parameter_summary.json")
label, sel = tab.T6_SECTIONS[0][1][0]          # "unrestr. Gaussian"
published = tab.select_theta(summary, sel)

cols = ["η₀", "η₁", "μ₀", "μ₁", "μ₂", "σ₀", "σ₁", "σ₂", "α₁", "α₂", "γ₁", "γ₂"]
rows = [("DGP", summary["truth"]["parameters"]),
        ("(1) unrestr. Gaussian — published", published),
        ("(1) unrestr. Gaussian — this run", mine)]
lines = ["| | " + " | ".join(cols) + " |", "|---" * (len(cols) + 1) + "|"]
for name, theta in rows:
    lines.append(f"| {name} | " + " | ".join(f"{v:.2f}" for v in tab.transform_theta(theta)) + " |")
display(Markdown("\n".join(lines)))

diff = max(abs(a - b) for a, b in zip(tab.transform_theta(published), tab.transform_theta(mine)))
print(f"largest absolute difference with the published row: {diff:.4f}")